# Unsupervised Learning: Clustering and Dimensionality Reduction

This notebook covers three core unsupervised techniques:
1. **K-Means** clustering
2. **PCA** (Principal Component Analysis) for dimensionality reduction
3. **DBSCAN** for density-based clustering

We apply them to real data and compare their strengths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

## 1. K-Means Clustering

K-Means partitions $n$ points into $k$ clusters by minimising the **within-cluster sum of squares**:
$$\min_{C_1,\ldots,C_k} \sum_{j=1}^{k} \sum_{x \in C_j} \|x - \mu_j\|^2$$

In [ ]:
# Load and scale data
wine = load_wine()
X = StandardScaler().fit_transform(wine.data)
y_true = wine.target

# Elbow method: find optimal k
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of clusters k')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
# Fit K-Means with k=3 (we know Wine has 3 classes)
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
labels_km = km3.labels_

print(f"Silhouette score: {silhouette_score(X, labels_km):.3f}")
print(f"Adjusted Rand Index vs true labels: {adjusted_rand_score(y_true, labels_km):.3f}")

## 2. PCA -- Dimensionality Reduction

PCA finds the orthogonal directions of maximum variance.
We project to 2D for visualisation and inspect explained variance.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained: {pca.explained_variance_ratio_.sum():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='Set1', s=30, edgecolors='k')
axes[0].set_title('PCA -- True Labels')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_km, cmap='Set2', s=30, edgecolors='k')
axes[1].set_title('PCA -- K-Means Labels')
for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

## 3. DBSCAN -- Density-Based Clustering

DBSCAN does **not** require specifying $k$. It groups points that are closely packed (high density) and marks outliers. Key parameters: `eps` (neighbourhood radius) and `min_samples`.

In [ ]:
# DBSCAN on non-convex data where K-Means fails
X_moons, y_moons = make_moons(n_samples=500, noise=0.08, random_state=42)

km_moons = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moons)
db_moons = DBSCAN(eps=0.15, min_samples=5).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons.labels_, cmap='Set1', s=15)
axes[0].set_title('K-Means on Moons')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_moons.labels_, cmap='Set1', s=15)
axes[1].set_title('DBSCAN on Moons')
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Pros | Cons |
|--------|------|------|
| K-Means | Fast, scalable | Needs $k$, assumes convex clusters |
| PCA | Linear, efficient | Only captures linear variance |
| DBSCAN | No $k$ needed, finds arbitrary shapes | Sensitive to `eps`, struggles with varying density |

**Next:** Model evaluation and selection.